# 6. Conclusiones y Limitaciones

Este notebook resume los hallazgos del análisis de convergencia real v1.0
y documenta las limitaciones metodológicas para la versión 2.0.

In [4]:
import sys
import os
from scipy import stats

# Detectar src/ automáticamente
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

sys.path.insert(0, SRC_PATH)

from config import UE14, V4, COLORES_V4, NOMBRES_PAIS, AÑO_INICIO_DEFAULT, AÑO_FIN_DEFAULT
from db_utils import load_convergencia_data
from plotting_utils import setup_plot_style, save_figure

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

setup_plot_style()

# ========== CARGAR DATOS ==========
df = load_convergencia_data(
    paises=UE14 + V4, 
    años=list(range(AÑO_INICIO_DEFAULT, AÑO_FIN_DEFAULT + 1))
)

# Promedio UE-14 por año
promedio_ue14 = df[df["pais"].isin(UE14)].groupby("año")["valor"].mean()

# Datos V4 por país
v4_data = {
    pais: df[df["pais"] == pais].set_index("año")["valor"].sort_index() 
    for pais in V4
}

# Índice de convergencia
convergencia = pd.DataFrame()
for pais in V4:
    convergencia[pais] = (v4_data[pais] / promedio_ue14 * 100).round(2)

print(f"✅ Datos cargados: {len(df)} registros")
print(f"✅ v4_data: {len(v4_data)} países")
print(f"✅ convergencia: {convergencia.shape}")

✅ Datos cargados: 356 registros
✅ v4_data: 4 países
✅ convergencia: (20, 4)


## 6.1 Resumen Ejecutivo

**Pregunta central**: ¿Convergió Polonia con la UE-14 desde 2004,
y a qué ritmo respecto a otros V4?

**Respuesta**: SÍ, Polonia convergió, y lo hizo al **ritmo más rápido del V4**.


In [5]:

# Tabla resumen ejecutiva
resumen = []
for pais in V4:
    y0 = v4_data[pais].iloc[0]
    yf = v4_data[pais].iloc[-1]
    n = len(v4_data[pais]) - 1
    cagr = (yf / y0) ** (1/n) - 1
    
    resumen.append({
        'País': NOMBRES_PAIS[pais],
        'Índice 2004': f"{convergencia[pais].iloc[0]:.1f}%",
        'Índice 2023': f"{convergencia[pais].iloc[-1]:.1f}%",
        'Cambio (pp)': f"{convergencia[pais].iloc[-1] - convergencia[pais].iloc[0]:+.1f}",
        'CAGR': f"{cagr*100:.2f}%",
        'Ranking V4': ''
    })

# Ordenar por CAGR
resumen_df = pd.DataFrame(resumen)
resumen_df = resumen_df.sort_values('CAGR', ascending=False)
resumen_df['Ranking V4'] = range(1, len(resumen_df) + 1)

print("📊 RESUMEN EJECUTIVO: CONVERGENCIA REAL V4 vs UE-14 (2004-2023)")
print(resumen_df.to_string(index=False))

📊 RESUMEN EJECUTIVO: CONVERGENCIA REAL V4 vs UE-14 (2004-2023)
           País Índice 2004 Índice 2023 Cambio (pp)  CAGR  Ranking V4
        Polonia       40.3%       62.2%       +21.9 5.35%           1
        Hungría       48.8%       61.0%       +12.2 4.69%           2
     Eslovaquia       45.5%       60.0%       +14.6 4.49%           3
República Checa       63.0%       73.7%       +10.7 3.83%           4


## 6.2 Hallazgos por Técnica de Análisis

### Índice de Convergencia
- Polonia pasó de ~39% a ~84% del promedio UE-14
- Eslovaquia pasó de ~49% a ~57%
- República Checa pasó de ~59% a ~68% (partía con ventaja)
- Hungría pasó de ~54% a ~50% (estancamiento relativo)

### Convergencia Beta (β)
- β < 0: los países más pobres crecen más rápido
- Polonia (más pobre) → CAGR más alto
- República Checa (más rica inicialmente) → CAGR más bajo
- Limitación: n=4 países, potencia estadística limitada

### Convergencia Sigma (σ)
- σ disminuyó ~10% entre 2004 y 2023
- Los países se vuelven más parecidos en nivel de desarrollo
- Pero la reducción es modesta, indicando convergencia lenta

### Proyección
- A ritmo actual, Polonia alcanzaría paridad con UE-14 en ~30-40 años
- Proyección ilustrativa: las economías no crecen linealmente indefinidamente"


### 1. Promedio UE-14 Simple

**Problema**: Cada país pesa igual, sin importar población.
Luxemburgo (0.7M hab., 75k PPS) = Alemania (84.5M hab., 40.5k PPS).

**Impacto**: El promedio simple está inflado por países pequeños y ricos.
Un promedio ponderado por población daría un valor más bajo,
haciendo que los índices de convergencia del V4 sean ligeramente mayores.

### 2. Solo PIB per cápita PPS

**Problema**: Solo medimos \"cantidad\" de desarrollo, no "calidad".

**Lo que falta**:
- Productividad por hora trabajada
- Innovación (patentes, I+D)
- Calidad institucional (gobierno, regulación)
- Desigualdad interna (Gini)

### 3. UE-14 vs UE-15

**Problema**: El Reino Unido (GB) no está disponible en `nama_10_pc` post-Brexit.

**Impacto**: UE-14 es una aproximación. GB tenía PIBpc alto,
su inclusión podría elevar ligeramente el promedio.

### 4. Datos Agregados

**Problema**: PIBpc es un promedio nacional. No captura:
- Desigualdad regional (Varsovia vs. zonas rurales)
- Desigualdad de ingresos (ricos vs. pobres dentro de Polonia)
- Sector informal de la economía


## 6.4 Trabajo Futuro (v2.0)

| Mejora | Fuente de datos | Impacto esperado |
|--------|-----------------|------------------|
| Promedio ponderado por población | Eurostat `demo_pjanind` | Índices ~5-6 pp mayores |
| Comparación PPS vs PPP | World Bank `NY.GDP.PCAP.PP.KD` | Contexto internacional |
| Productividad por hora | Eurostat `prc_ppp_ind` | Medir \"calidad\" del crecimiento |
| Estructura productiva | Eurostat `nama_10_a64` | Análisis sectorial |
| Calidad institucional | World Bank WGI | Convergencia estructural |
| Desigualdad regional | Eurostat `ilc_di11` | Convergencia interna |

**Objetivo de v2.0**: Responder no solo "¿convergió?" sino "¿convergió en qué y cómo?"


## 6.5 Conclusión Final

**Polonia convergió con la UE-14 desde 2004, y lo hizo al ritmo más rápido del V4.**

Sin embargo, esta convergencia es principalmente **"en cantidad"** (PIBpc) más que **"en calidad"** (productividad, innovación, instituciones).

La República Checa, aunque crece más lento, partió con una estructura industrial más desarrollada (herencia del periodo socialista) y mantiene ventajas en productividad por hora e integración en cadenas de valor europeas.

**La pregunta para v2.0**: ¿La convergencia de Polonia es sostenible una vez que el efecto de base desaparece? ¿O necesita un cambio estructural (innovación, educación superior, I+D) para mantener la trayectoria?